# 05. Relational Database Serialization (MySQL Ingestion)

### Project: E-Commerce Customer Funnel & Cohort Retention Analytics

**Objective:** Ingest sanitized fact and dimension tables into a structured MySQL relational schema to enable downstream ad-hoc analytical querying and DirectQuery / import connectivity for Power BI.

## 1. Connection Configuration & Schema Verification

In [ ]:
import warnings
import pandas as pd
from sqlalchemy import create_engine, text, URL

warnings.filterwarnings('ignore')

MYSQL_USER     = 'root'
MYSQL_PASSWORD = '************'
MYSQL_HOST     = 'localhost'
MYSQL_PORT     = 3306
MYSQL_DATABASE = 'retail_analysis'

# Initialize server connection
server_url = URL.create(
    drivername='mysql+mysqlconnector',
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT
)
server_engine = create_engine(server_url)

# Ensure target analytical schema exists
with server_engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {MYSQL_DATABASE};"))
    conn.commit()
    print(f'Database schema "{MYSQL_DATABASE}" verified.')

# Connect to target database
db_url = URL.create(
    drivername='mysql+mysqlconnector',
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE
)
engine = create_engine(db_url)
print('Database engine initialized successfully.')

Database schema "retail_analysis" verified.
Database engine initialized successfully.


## 2. Ingestion of Clean Fact and Dimension Tables

In [2]:
print('Loading serialized tables...')
df_fact = pd.read_csv('data/retail_cleaned.csv')
df_customers = pd.read_csv('data/customer_summary.csv')
df_cohorts = pd.read_csv('data/cohort_counts.csv')

print(f'- Fact Transactions : {len(df_fact):,} rows')
print(f'- Customer Dimension: {len(df_customers):,} rows')
print(f'- Cohort Dimension  : {len(df_cohorts):,} rows')

# Ingest tables in batches
print('\nWriting tables to MySQL...')
df_fact.to_sql('retail_clean', con=engine, if_exists='replace', index=False, chunksize=1000)
df_customers.to_sql('customer_summary', con=engine, if_exists='replace', index=False, chunksize=1000)
df_cohorts.to_sql('cohort_counts', con=engine, if_exists='replace', index=False, chunksize=1000)
print('Ingestion completed successfully.')

Loading serialized tables...
- Fact Transactions : 779,425 rows
- Customer Dimension: 5,878 rows
- Cohort Dimension  : 325 rows

Writing tables to MySQL...
Ingestion completed successfully.


## 3. Database Ingestion Audit

In [3]:
with engine.connect() as conn:
    c_fact = conn.execute(text('SELECT COUNT(*) FROM retail_clean')).fetchone()[0]
    c_cust = conn.execute(text('SELECT COUNT(*) FROM customer_summary')).fetchone()[0]
    c_coho = conn.execute(text('SELECT COUNT(*) FROM cohort_counts')).fetchone()[0]

print('Relational Database Record Counts:')
print(f'- retail_clean     : {c_fact:,}')
print(f'- customer_summary : {c_cust:,}')
print(f'- cohort_counts    : {c_coho:,}')

Relational Database Record Counts:
- retail_clean     : 779,425
- customer_summary : 5,878
- cohort_counts    : 325
